In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.utils import save_image
import matplotlib.pyplot as plt
from PIL import Image
import kagglehub
from tqdm import tqdm

# --- 1. 环境与参数设置 ---
print("Step 1: Setting up environment and hyperparameters...")
dataset_path = kagglehub.dataset_download("splcher/animefacedataset")
image_folder_path = os.path.join(dataset_path, 'images')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
IMG_SIZE = 64
LATENT_DIM = 100
EPOCHS = 50
LEARNING_RATE = 1e-3
BETA = 1
# 新增：Checkpoint设置
CHECKPOINT_DIR = "vae_checkpoints"
SAVE_EVERY = 10 # 每10个epoch保存一次

print(f"Using device: {DEVICE}")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# --- 2. 数据加载与预处理 ---
print("\nStep 2: Loading ALL images into a single GPU tensor...")
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
image_files = [os.path.join(image_folder_path, f) for f in os.listdir(image_folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
tensor_list = [transform(Image.open(img_path).convert("RGB")) for img_path in tqdm(image_files, desc="Loading images")]
full_dataset_tensor = torch.stack(tensor_list).to(DEVICE)
dataset_size_gb = full_dataset_tensor.nelement() * full_dataset_tensor.element_size() / (1024**3)
print(f"Data loading complete. Shape: {full_dataset_tensor.shape}, Size on GPU: {dataset_size_gb:.2f} GB")


# --- 3. 模型构建 (Unchanged) ---
print("\nStep 3: Building the VAE model...")
class VAE(nn.Module):
    # (模型代码与之前完全相同)
    def __init__(self, latent_dim=100):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_log_var = nn.Linear(256 * 4 * 4, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1), nn.Tanh()
        )
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_log_var(h)
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 256, 4, 4)
        return self.decoder(h)
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

# --- 4. 损失函数定义 (Unchanged) ---
print("Step 4: Defining the loss function...")
def loss_function(recon_x, x, mu, log_var):
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + BETA * kld

# --- 5. 训练准备与循环 ---
print("\nStep 5: Preparing for training...")
model = VAE(latent_dim=LATENT_DIM).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 新增：加载Checkpoint
start_epoch = 0
checkpoint_path = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Resuming training from epoch {start_epoch + 1}")

os.makedirs("vae_results/reconstruction", exist_ok=True)
os.makedirs("vae_results/generation", exist_ok=True)
num_images = full_dataset_tensor.shape[0]

# 新增：用于存储损失的列表
epoch_losses = []

print("\nStarting training...")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0
    shuffled_indices = torch.randperm(num_images)
    
    for i in tqdm(range(0, num_images, BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}"):
        indices = shuffled_indices[i : i + BATCH_SIZE]
        data = full_dataset_tensor[indices]
        recon_batch, mu, log_var = model(data)
        loss = loss_function(recon_batch, data, mu, log_var)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_loss = train_loss / num_images
    epoch_losses.append(avg_loss) # 记录当前epoch的平均损失
    print(f"====> Epoch: {epoch+1} | Average loss: {avg_loss:.4f}")

    # ... (评估和保存图像的代码不变) ...
    model.eval()
    with torch.no_grad():
        sample_data = full_dataset_tensor[:8]
        recon_sample, _, _ = model(sample_data)
        comparison = torch.cat([sample_data, recon_sample])
        save_image(comparison.cpu(), f"vae_results/reconstruction/recon_{epoch+1}.png", nrow=8, normalize=True)
        z_sample = torch.randn(64, LATENT_DIM).to(DEVICE)
        generated_sample = model.decode(z_sample).cpu()
        save_image(generated_sample, f"vae_results/generation/gen_{epoch+1}.png", normalize=True)
    
    # 新增：保存Checkpoint
    if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
        checkpoint_save_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pth")
        latest_save_path = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")
        print(f"Saving checkpoint to {checkpoint_save_path}")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, checkpoint_save_path)
        # 更新latest_checkpoint
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, latest_save_path)


print("Training finished!")

Step 1: Setting up environment and hyperparameters...
Using device: cuda

Step 2: Loading ALL images into a single GPU tensor...


Loading images: 100%|██████████| 63565/63565 [00:29<00:00, 2119.33it/s]


Data loading complete. Shape: torch.Size([63565, 3, 64, 64]), Size on GPU: 2.91 GB

Step 3: Building the VAE model...
Step 4: Defining the loss function...

Step 5: Preparing for training...

Starting training...


Epoch 1/50: 100%|██████████| 497/497 [00:01<00:00, 271.11it/s]


====> Epoch: 1 | Average loss: 1519.0797


...

Epoch 49/50: 100%|██████████| 497/497 [00:01<00:00, 272.22it/s]


====> Epoch: 49 | Average loss: 676.1993


Epoch 50/50: 100%|██████████| 497/497 [00:01<00:00, 275.63it/s]


====> Epoch: 50 | Average loss: 675.2712
Saving checkpoint to vae_checkpoints\checkpoint_epoch_50.pth
Training finished!

Step 6: Visualizing results...


: 